# Поиск по документам: TF-IDF и SVD

На прошлом занятии сообщение получало название отдела. Теперь пользователю нужен конкретный документ: например, инструкция по продлению книги. Построим два варианта поиска по одной базе и посмотрим, когда изменение представления меняет порядок результатов.

В базе двенадцать документов. Вопросы хранятся отдельно; часть из них относится к сведениям, которых в документах нет.

In [2]:
from pathlib import Path
import os
import json
import re
import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Находим корень раздачи независимо от места запуска Jupyter.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data" / "requests.csv").exists())
DATA = ROOT / "data"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
np.random.seed(42)
plt.rcParams.update({"font.family": "DejaVu Sans", "figure.figsize": (8, 4)})


## Что будем искать

Откройте список документов и выберите тот, в котором ожидаете найти условия продления книги. Позже сравним это предположение с выдачей. Поле `relevant_doc` в таблице вопросов содержит эталонный документ; пустая строка означает, что ответа в базе нет. Эти метки используются при оценке, а не при построении индекса.

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity

docs = pd.read_csv(DATA / "knowledge.csv", keep_default_na=False)
queries = pd.read_csv(DATA / "retrieval_queries.csv", keep_default_na=False)
content = docs["title"] + ". " + docs["text"]
doc_ids = docs["doc_id"].to_numpy()
assert docs["doc_id"].is_unique
assert not {"relevant_doc", "reference_answer"} & set(docs.columns)
print("Документов:", len(docs), "Вопросов:", len(queries))
display(docs[["doc_id", "title"]])

Документов: 12 Вопросов: 30


,doc_id,title
0,D01,Восстановление доступа
1,D02,Техническая поддержка
2,D03,Продление книг
3,D04,Читальный зал
4,D05,Заселение
5,D06,Ремонт в комнате
6,D07,Пересдача
7,D08,Исправление ведомости
8,D09,Документы абитуриента
9,D10,Консультации поступающим


### Один расчёт перед функцией

Пусть запрос представлен вектором (1;0), а документ — (1;1). Их скалярное произведение равно 1, а произведение длин равно √2. Следующая ячейка воспроизводит расчёт. Увеличьте второй вектор в десять раз и сначала предскажите результат.

In [4]:
demo_query = np.array([1., 0.])
demo_document = np.array([1., 1.])
print(np.dot(demo_query, demo_document) / (np.linalg.norm(demo_query) * np.linalg.norm(demo_document)))

0.7071067811865475


## 1. Сходство направлений

Напишите `cosine_safe(a, b)` для одномерных векторов одинаковой размерности. Используйте скалярное произведение и евклидовы нормы. Если размерности не совпадают или одна из норм равна нулю, вызывайте `ValueError`.

Сравните (1;0) с (1;1), (0;1), (10;10) и (−1;0). Объясните, почему положительный масштаб не влияет на косинус, а отрицательный меняет его знак. Можно ли прочитать число 0,7071 как вероятность правильного ответа?

In [5]:
def cosine_safe(a, b):
    # Ваш код
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)

    if a.ndim != 1 or b.ndim != 1:
        raise ValueError("нужны одномерные векторы")
    if a.shape != b.shape:
        raise ValueError("размерности векторов не совпадают")

    norm_a = np.linalg.norm(a) 
    norm_b = np.linalg.norm(b)
    if norm_a == 0 or norm_b == 0:
        raise ValueError("косинус для нулевого вектора не определён")

    return np.dot(a, b) / (norm_a * norm_b)


base = [1, 0]
for other in [[1, 1], [0, 1], [10, 10], [-1, 0]]:
    print(base, "&", other, "->", round(cosine_safe(base, other), 4))

[1, 0] & [1, 1] -> 0.7071
[1, 0] & [0, 1] -> 0.0
[1, 0] & [10, 10] -> 0.7071
[1, 0] & [-1, 0] -> -1.0


In [6]:
assert np.isclose(cosine_safe([1, 0], [1, 1]), 1 / np.sqrt(2))
assert np.isclose(cosine_safe([1, 0], [10, 10]), cosine_safe([1, 0], [1, 1]))
try:
    cosine_safe([0, 0], [1, 0])
except ValueError:
    pass
else:
    raise AssertionError("Нужна явная обработка нулевого вектора")

**Ответ.** (1;0) и (1;1): 0,7071; (1;0) и (0;1): 0 — векторы перпендикулярны; (1;0) и (10;10): снова 0,7071; (1;0) и (−1;0): −1 — противоположные направления.

Положительный множитель c увеличивает и скалярное произведение, и длину вектора в c раз, поэтому в дроби он сокращается. При отрицательном множителе числитель меняет знак, а длина остаётся положительной (|c|), поэтому меняется знак косинуса.

## 2. Два представления одной базы

Обучите `vectorizer = TfidfVectorizer(ngram_range=(1, 2), norm="l2")` на `content` и сохраните матрицу в `X_docs`. Затем обучите `svd = TruncatedSVD(n_components=6, random_state=42)` на этой матрице. Нормируйте полученные строки через `normalize` и сохраните их в `Z_docs`.

Сравните формы матриц и количество ненулевых элементов. Объясните, что теперь означает строка и что изменилось в её координатах. Почему для индекса можно использовать все известные документы, но нельзя добавлять в обучение проверочные вопросы с ответами?

In [8]:
# Ваш код
vectorizer = TfidfVectorizer(ngram_range=(1, 2), norm="l2")
X_docs = vectorizer.fit_transform(content)

svd = TruncatedSVD(n_components=6, random_state=42)
Z_raw = svd.fit_transform(X_docs)

Z_docs = normalize(Z_raw)

print("X_docs:", X_docs.shape, "ненулевых элементов:", X_docs.nnz)
print("Z_docs:", Z_docs.shape, "ненулевых элементов:", np.count_nonzero(Z_docs))

X_docs: (12, 670) ненулевых элементов: 816
Z_docs: (12, 6) ненулевых элементов: 72


In [7]:
assert X_docs.shape[0] == len(docs)
assert Z_docs.shape == (len(docs), 6)
assert np.allclose(np.linalg.norm(Z_docs, axis=1), 1)
assert svd.components_.shape[1] == len(vectorizer.vocabulary_)

**Ответ.** `X_docs` имеет форму 12×670: 12 документов и 670 признаков (слова и пары слов), но ненулевых всего 816 из 8040 — матрица разреженная. `Z_docs` имеет форму 12×6, и все 72 элемента ненулевые — матрица плотная.

Строка по-прежнему означает один документ (та же позиция, тот же `doc_id`). В TF-IDF каждая координата — вес конкретного слова; в SVD координата — проекция на одно из шести направлений, найденных по всей матрице. Такое направление смешивает много слов сразу, и у него нет простого названия.

Индекс можно обучать на всех двенадцати документах: это заранее известная база, по которой системе и разрешено искать. А вопросы с эталонными ответами — это проверка. Если добавить их в обучение, словарь и направления SVD подстроятся под проверочные формулировки, и метрики покажут лучший результат, чем будет на настоящих новых запросах.

## 3. От запроса к выдаче

Закончите `search(query, method="tfidf", k=3)`. Запрос должен проходить через уже обученные `vectorizer` и, для метода `svd`, `svd` с последующей нормировкой. Сравните его с соответствующей матрицей документов.

Верните DataFrame с колонками `doc_id`, `title`, `score`: сначала большая неокруглённая оценка, при равенстве — меньший `doc_id`. Для нулевого вектора запроса верните пустую таблицу с теми же колонками. Проверьте оба метода на вопросе «Как продлить книгу?», пустой строке и строке «щщщщщщщ».

Почему новое `fit` на одном запросе испортило бы сравнение, даже если удалось получить массив нужного размера?

In [10]:
def search(query, method="tfidf", k=3):
    # Ваш код
    columns = ["doc_id", "title", "score"]
    query_tfidf = vectorizer.transform([query]).toarray()[0]

    if method == "tfidf":
        query_vector = query_tfidf
        doc_matrix = X_docs.toarray()
    elif method == "svd":
        query_vector = svd.transform([query_tfidf])[0]
        doc_matrix = Z_docs
    else:
        raise ValueError("method должен быть 'tfidf' или 'svd'")

    query_norm = np.linalg.norm(query_vector)
    if query_norm < 1e-12:
        return pd.DataFrame(columns=columns)

    query_vector = query_vector / query_norm
    scores = doc_matrix @ query_vector

    result = pd.DataFrame({
        "doc_id": doc_ids,
        "title": docs["title"],
        "score": scores,
    })
    result = result.sort_values(["score", "doc_id"], ascending=[False, True])
    return result.head(k).reset_index(drop=True)


for text in ["Как продлить книгу?", "", "щщщщщщщ"]:
    for method in ["tfidf", "svd"]:
        print(repr(text), method)
        display(search(text, method, 3))

'Как продлить книгу?' tfidf


,doc_id,title,score
0,D03,Продление книг,0.252971
1,D01,Восстановление доступа,0.000000
2,D02,Техническая поддержка,0.000000


'Как продлить книгу?' svd


,doc_id,title,score
0,D03,Продление книг,0.994567
1,D08,Исправление ведомости,0.550999
2,D01,Восстановление доступа,0.452946


'' tfidf


...

,doc_id,title,score


'щщщщщщщ' tfidf


,doc_id,title,score


'щщщщщщщ' svd


,doc_id,title,score


In [11]:
for method in ["tfidf", "svd"]:
    r = search("Как продлить книгу?", method, 3)
    assert list(r.columns) == ["doc_id", "title", "score"]
    assert len(r) <= 3 and r["doc_id"].is_unique
    assert r["score"].is_monotonic_decreasing
    assert search("", method).empty
    assert search("щщщщщщщ", method).empty

**Ответ.** Для «Как продлить книгу?» оба метода ставят первым D03 «Продление книг». У TF-IDF второе и третье места имеют оценку 0: это не совпадения, а просто следующие документы по правилу сортировки `doc_id`. Пустая строка и «щщщщщщщ» не содержат слов из словаря, вектор запроса нулевой, поэтому функция возвращает пустую таблицу.

Новый `fit` на одном запросе построил бы другой словарь и другие направления SVD. Номер координаты у запроса и у документов означал бы разные вещи: например, третья координата запроса — слово «книгу», а третья координата документа — совсем другое слово. Размер массива можно подогнать, но сравнение получится бессмысленным. Поэтому для запроса вызывается только `transform` уже обученных `vectorizer` и `svd`.

## 4. Считаем попадания и ранги

Реализуйте `evaluate(part, method)`. В среднее включайте только вопросы с непустым `relevant_doc`. Возвращайте словарь с ключами `recall_at_1`, `recall_at_3`, `mrr_at_3`, `n_answerable`. Если эталон отсутствует в первых трёх результатах, его вклад в MRR@3 равен нулю.

Соберите `results_table` с колонками `split`, `method` и четырьмя показателями. Сравните заранее выбранные TF-IDF и SVD(6) на validation, затем на test. После просмотра теста оставьте настройки неизменными.

Проверьте формулы на четырёх рангах из лекции: 1, 3, отсутствие в top-3 и 2. Чем отличаются Recall@3 и MRR@3 на этом примере?

In [12]:
def rank_of(relevant_doc, found_doc_ids):
    """Место эталона в выдаче (1, 2, 3) или None, если его там нет."""
    if relevant_doc in found_doc_ids:
        return found_doc_ids.index(relevant_doc) + 1
    return None


def metrics_from_ranks(ranks):
    """Recall@1, Recall@3 и MRR@3 по списку рангов эталона."""
    hits_at_1 = []
    hits_at_3 = []
    reciprocal_ranks = []
    for rank in ranks:
        hits_at_1.append(1 if rank == 1 else 0)
        hits_at_3.append(1 if rank is not None and rank <= 3 else 0)
        if rank is not None and rank <= 3:
            reciprocal_ranks.append(1 / rank)
        else:
            reciprocal_ranks.append(0)
    return {
        "recall_at_1": np.mean(hits_at_1),
        "recall_at_3": np.mean(hits_at_3),
        "mrr_at_3": np.mean(reciprocal_ranks),
    }


def evaluate(part, method):
    # Ваш код
    answerable = part[part["relevant_doc"] != ""]

    ranks = []
    for row in answerable.itertuples():
        found = search(row.query, method, 3)["doc_id"].tolist()
        ranks.append(rank_of(row.relevant_doc, found))

    scores = metrics_from_ranks(ranks)
    scores["n_answerable"] = len(answerable)
    return scores

# Ваш код
rows = []
for split in ["validation", "test"]:
    part = queries[queries["split"] == split]
    for method in ["tfidf", "svd"]:
        row = {"split": split, "method": method}
        row.update(evaluate(part, method))
        rows.append(row)
results_table = pd.DataFrame(rows)
display(results_table)

print("Пример из лекции:", metrics_from_ranks([1, 3, None, 2]))

,split,method,recall_at_1,recall_at_3,mrr_at_3,n_answerable
0,validation,tfidf,0.750,0.8750,0.791667,8
1,validation,svd,0.875,0.8750,0.875000,8
2,test,tfidf,0.875,0.9375,0.906250,16
3,test,svd,0.875,1.0000,0.916667,16


Пример из лекции: {'recall_at_1': np.float64(0.25), 'recall_at_3': np.float64(0.75), 'mrr_at_3': np.float64(0.4583333333333333)}


In [13]:
assert (results_table["recall_at_3"] >= results_table["recall_at_1"] - 1e-12).all()
assert results_table["mrr_at_3"].between(0, 1).all()
assert set(results_table["n_answerable"]) == {8, 16}
# Четыре ранга из лекции: 1, 3, отсутствие и 2.
assert np.isclose(np.mean([1, 1/3, 0, 1/2]), 11/24)


**Ответ.** На validation (8 ответимых вопросов) SVD лучше по Recall@1 (0,875 против 0,75) и MRR@3 (0,875 против 0,792), а Recall@3 одинаковый — 0,875. На test (16 вопросов) Recall@1 одинаковый — 0,875, а Recall@3 у SVD 1,0 против 0,9375 у TF-IDF: TF-IDF один раз не нашёл эталон в top-3. Настройки после теста не менялись.

Пример из лекции, ранги 1, 3, «нет» и 2: Recall@1 = 1/4 = 0,25, Recall@3 = 3/4 = 0,75, MRR@3 = (1 + 1/3 + 0 + 1/2)/4 ≈ 0,4583. Recall@3 засчитывает 2-е и 3-е места так же, как первое. MRR@3 штрафует за низкое место: 3-е место даёт только 1/3. Если переставить эталон с 1-го места на 3-е, Recall@3 не изменится, а MRR@3 уменьшится.

## Читаем различия выдач

Следующая ячейка собирает вопросы, для которых порядок документов отличается, и показывает оценки. Выберите два таких вопроса для разбора. Если документ попал в выдачу с нулевой оценкой, проверьте роль сортировки при равенстве.

После таблицы появятся вопросы без ответа в базе. Поиск всё равно может вернуть близкий документ. Прочитайте его и найдите, какого именно факта не хватает. Особенно полезен вопрос о сумме стипендии.

In [16]:
differences = []
for row in queries[queries["relevant_doc"].ne("")].itertuples():
    first = search(row.query, "tfidf", 3)
    second = search(row.query, "svd", 3)
    if first["doc_id"].tolist() != second["doc_id"].tolist():
        differences.append({"query_id": row.query_id, "query": row.query,
                            "relevant_doc": row.relevant_doc,
                            "tfidf": list(zip(first["doc_id"], first["score"].round(6))),
                            "svd": list(zip(second["doc_id"], second["score"].round(6)))})
differences_table = pd.DataFrame(differences)
display(differences_table)

print("Вопросы без ответа в базе:")
for row in queries[queries["relevant_doc"].eq("")].itertuples():
    print(row.query)
    for method in ["tfidf", "svd"]:
        r = search(row.query, method, 1)
        print(method, r.to_dict("records"))

,query_id,query,relevant_doc,tfidf,svd
0,Q01,Как долго действует ссылка для смены пароля?,D01,"[(D01, 0.149885), (D04, 0.049506), (D11, 0.028...","[(D01, 0.923058), (D02, 0.607284), (D03, 0.433..."
1,Q02,"Что проверить, если письмо восстановления не п...",D01,"[(D01, 0.250032), (D03, 0.043662), (D08, 0.034...","[(D01, 0.978102), (D03, 0.631905), (D02, 0.329..."
2,Q03,Во сколько заканчивает работу служба ИТ?,D02,"[(D02, 0.191967), (D01, 0.050861), (D03, 0.0)]","[(D02, 0.933161), (D04, 0.785474), (D01, 0.619..."
3,Q04,Какие сведения нужны при технической ошибке?,D02,"[(D05, 0.085492), (D01, 0.080678), (D02, 0.0)]","[(D02, 0.789882), (D01, 0.783168), (D04, 0.623..."
4,Q05,На сколько дней можно продлить книгу?,D03,"[(D03, 0.327433), (D01, 0.056645), (D07, 0.047...","[(D03, 0.98348), (D01, 0.596384), (D08, 0.4862..."
5,Q06,"Можно ли продлить издание, на которое есть бронь?",D03,"[(D03, 0.258971), (D01, 0.059795), (D07, 0.020...","[(D03, 0.976615), (D01, 0.634482), (D08, 0.469..."
6,Q07,До какого часа открыт читальный зал?,D04,"[(D04, 0.411249), (D07, 0.022403), (D10, 0.022...","[(D04, 0.995338), (D02, 0.922696), (D05, 0.899..."
7,Q08,Разрешено ли брать редкие издания домой?,D04,"[(D04, 0.205364), (D01, 0.0), (D02, 0.0)]","[(D04, 0.993395), (D02, 0.946425), (D05, 0.900..."
8,Q09,Кто выдаёт ключи при заселении?,D05,"[(D05, 0.139608), (D01, 0.065873), (D02, 0.0)]","[(D02, 0.912866), (D04, 0.846822), (D05, 0.811..."
9,Q10,Во сколько заканчивается заселение в рабочий д...,D05,"[(D05, 0.256477), (D04, 0.08384), (D01, 0.0)]","[(D05, 0.993838), (D04, 0.922965), (D02, 0.828..."


Вопросы без ответа в базе:
Какой проходной балл в этом году?
tfidf [{'doc_id': 'D07', 'title': 'Пересдача', 'score': 0.12838630228462158}]
svd [{'doc_id': 'D07', 'title': 'Пересдача', 'score': 0.9891558241766476}]
Сколько рублей составляет академическая стипендия?
tfidf [{'doc_id': 'D11', 'title': 'Социальная стипендия', 'score': 0.13559796286874265}]
svd [{'doc_id': 'D11', 'title': 'Социальная стипендия', 'score': 0.990860123957971}]
Когда у группы ИВТ-42 экзамен по физике?
tfidf [{'doc_id': 'D11', 'title': 'Социальная стипендия', 'score': 0.08372475602874817}]
svd [{'doc_id': 'D04', 'title': 'Читальный зал', 'score': 0.9657118976678085}]
Какова стоимость проживания за месяц?
tfidf [{'doc_id': 'D05', 'title': 'Заселение', 'score': 0.10287431176909809}]
svd [{'doc_id': 'D10', 'title': 'Консультации поступающим', 'score': 0.8042034216625412}]
Как зовут директора библиотеки?
tfidf []
svd []
Какая сегодня погода у университета?
tfidf [{'doc_id': 'D11', 'title': 'Социальная стипендия', 'sc

## Разбор выдачи

Выберите два вопроса, для которых методы дали разный порядок документов, и один вопрос без ответа в базе. Для каждого прочитайте найденные документы: где именно находится нужное сведение и какие совпадения могли повлиять на ранжирование? Покажите, как изменение ранга отражается на метриках. Если на выбранных вопросах выдачи совпали, возьмите другую строку из таблицы различий.

Сдайте выполненный ноутбук с этими разборами. Перед сдачей перезапустите ядро и выполните все ячейки по порядку.

### Мой разбор

**Q21 «Сколько рассматривают документы на социальную стипендию?» (test, эталон D11).** TF-IDF: D09, D03, D07 — эталона нет в top-3. Совпали только «документы» (D09) и «на» (D03). В D11 стоят другие формы: «социальной стипендии», «рассмотрения», поэтому точного совпадения слов нет. SVD ставит D11 первым (0,7218), но D09 почти рядом (0,7212). Ответ в D11: «Срок рассмотрения — 5 рабочих дней». Ранг «нет» → 1: вклад в MRR@3 и Recall@3 вырос с 0 до 1. Именно этот вопрос даёт разницу Recall@3 на test: 0,9375 против 1,0.

**Q09 «Кто выдаёт ключи при заселении?» (test, эталон D05) — здесь SVD хуже.** TF-IDF ставит D05 первым: совпали «ключи» и «выдаёт». SVD: D02 (0,913), D04 (0,847), D05 (0,812). Шесть направлений смешали точные слова, и документ про ИТ оказался ближе. Ранг 1 → 3: вклад в MRR@3 упал с 1 до 1/3. Сжатие помогает со словоформами, но может потерять редкие точные слова.

**Вопрос без ответа: Q26 «Сколько рублей составляет академическая стипендия?»** Оба метода находят D11 «Социальная стипендия», SVD — с косинусом 0,9909. Совпало одно слово «стипендия». В D11 говорится о социальной стипендии, составе заявления и сроке рассмотрения. Суммы в рублях нет, и об академической стипендии ничего не сказано. В D12 прямо написано, что конкретные суммы в базе не заданы. Высокая оценка показывает только близость темы и не означает, что ответ есть. Поэтому такие вопросы не входят в Recall и MRR: эталона у них нет. Для них нужен отдельный механизм, например порог оценки или проверка ответа по тексту. Заметно, что у SVD почти все оценки высокие (0,75–0,99) даже для вопроса о погоде, так что простой порог по SVD-косинусу здесь не работает.